# Spam Detection — Đóng gói tiền xử lý & Vectorizer
**Thành viên:** B — Kỹ thuật dữ liệu  
**Tuần 2:** Đóng gói hàm preprocess(), đảm bảo TF-IDF shape, lưu vectorizer  
**Đầu ra:** preprocessing functions + vectorizer.pkl

## 1. Import thư viện

In [1]:
import re
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd

## 2. Hàm tiền xử lý tái sử dụng

In [2]:
def preprocess(text):
    """Làm sạch một chuỗi văn bản.

    Các bước: lowercase → xóa ký tự đặc biệt → strip khoảng trắng.
    Khớp chính xác với clean_text() đã dùng để train vectorizer ở Tuần 1.
    """
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = text.strip()  # đồng bộ với clean_text() Tuần 1
    return text


def preprocess_list(texts):
    """Áp dụng preprocess() cho danh sách văn bản."""
    return [preprocess(t) for t in texts]


# Kiểm tra nhanh
samples = [
    '  FREE entry!! Win $100 today  ',
    'Hey how are you doing?',
    'URGENT: Click now to claim prize!!!',
]
for s in samples:
    print(repr(s), '->', repr(preprocess(s)))

'  FREE entry!! Win $100 today  ' -> 'free entry win  today'
'Hey how are you doing?' -> 'hey how are you doing'
'URGENT: Click now to claim prize!!!' -> 'urgent click now to claim prize'


## 3. Tải dữ liệu và fit vectorizer

In [3]:
df = pd.read_csv('clean_spam_dataset.csv')
print(f'Shape: {df.shape}')
print(f'Null trong clean_text: {df["clean_text"].isnull().sum()}')
df.head(3)

Shape: (5166, 3)
Null trong clean_text: 0


,label,text,clean_text
0,0,"Go until jurong point, crazy.. Available only ...",go until jurong point crazy available only in ...
1,0,Ok lar... Joking wif u oni...,ok lar joking wif u oni
2,1,Free entry in 2 a wkly comp to win FA Cup fina...,free entry in a wkly comp to win fa cup final...


In [4]:
# Fit TF-IDF trên cột clean_text (đã được preprocess từ Tuần 1)
vectorizer = TfidfVectorizer(max_features=5000)
X = vectorizer.fit_transform(df['clean_text'].fillna(''))
y = df['label']

print(f'Shape ma trận TF-IDF: {X.shape}')
print(f'Vocab size: {len(vectorizer.vocabulary_)}')
assert X.shape[1] == 5000, 'Shape sai!'
print('Shape OK — đúng (n_samples, 5000)')

Shape ma trận TF-IDF: (5166, 5000)
Vocab size: 5000
Shape OK — đúng (n_samples, 5000)


## 4. Hàm transform và lưu/load vectorizer

In [5]:
def transform(texts, vectorizer):
    """Transform văn bản bằng vectorizer đã fit (dùng khi inference).

    KHÔNG gọi fit lại — phải dùng vectorizer đã lưu để đảm bảo
    nhất quán với model của C.

    Args:
        texts     : list[str] — văn bản thô (chưa preprocess)
        vectorizer: TfidfVectorizer đã fit (load từ vectorizer.pkl)
    Returns:
        scipy sparse matrix shape (n_samples, 5000)
    """
    cleaned = preprocess_list(texts)
    return vectorizer.transform(cleaned)


def save_vectorizer(vectorizer, path='vectorizer.pkl'):
    """Lưu vectorizer ra file."""
    joblib.dump(vectorizer, path)
    print(f'Đã lưu: {path}')


def load_vectorizer(path='vectorizer.pkl'):
    """Load vectorizer từ file."""
    return joblib.load(path)


def fit_transform_new(texts):
    """Fit vectorizer MỚI và transform.

    CẢNH BÁO: Chỉ dùng khi train lại từ đầu.
    Khi deploy hoặc inference → dùng transform() + load_vectorizer().
    """
    cleaned = preprocess_list(texts)
    vec = TfidfVectorizer(max_features=5000)
    X = vec.fit_transform(cleaned)
    return X, vec

## 5. Lưu vectorizer bằng joblib

In [6]:
save_vectorizer(vectorizer, 'vectorizer.pkl')

# Kiểm tra load lại
vec_loaded = load_vectorizer('vectorizer.pkl')
print(f'Vocab size sau khi load: {len(vec_loaded.vocabulary_)}')
assert len(vec_loaded.vocabulary_) == 5000
print('Load OK!')

Đã lưu: vectorizer.pkl


Vocab size sau khi load: 5000
Load OK!


## 6. Kiểm tra transform() với dữ liệu mới

In [7]:
test_texts = [
    'Free prize! Click now to win cash reward',
    'Are we still meeting tomorrow at 9am?',
    '  URGENT!! Claim your FREE gift today!!!  ',
]

X_test = transform(test_texts, vec_loaded)
print(f'Shape: {X_test.shape}')
assert X_test.shape == (3, 5000), 'Shape sai!'

# In top 5 từ có TF-IDF cao nhất cho mỗi câu
feature_names = vec_loaded.get_feature_names_out()
for i, text in enumerate(test_texts):
    row = X_test[i].toarray()[0]
    top5_idx = row.argsort()[::-1][:5]
    top5 = [(feature_names[j], round(row[j], 4)) for j in top5_idx if row[j] > 0]
    print(f'\n[{i+1}] "{text.strip()[:40]}..."')
    print(f'     Top từ: {top5}')

Shape: (3, 5000)

[1] "Free prize! Click now to win cash reward..."
     Top từ: [('click', np.float64(0.4943)), ('reward', np.float64(0.4857)), ('win', np.float64(0.3615)), ('cash', np.float64(0.3515)), ('prize', np.float64(0.341))]

[2] "Are we still meeting tomorrow at 9am?..."
     Top từ: [('meeting', np.float64(0.4953)), ('tomorrow', np.float64(0.4369)), ('still', np.float64(0.3848)), ('am', np.float64(0.3556)), ('we', np.float64(0.3256))]

[3] "URGENT!! Claim your FREE gift today!!!..."
     Top từ: [('gift', np.float64(0.527)), ('urgent', np.float64(0.4512)), ('claim', np.float64(0.415)), ('today', np.float64(0.3879)), ('free', np.float64(0.3512))]
